In [1]:
# Birdset Imports
from datasets import Audio, load_from_disk, load_dataset
from extractors import Birdset

/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# SAM Imports
from sam_audio import SAMAudio, SAMAudioProcessor
import torch

/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# Fun Loading imports
from tqdm import tqdm

In [4]:
region = "HSN"

In [5]:
data = load_dataset("DBD-research-group/BirdSet", region, trust_remote_code=True)

In [6]:
# SAM Settings
model = "sam-audio-base"

In [7]:
# Setting up SAM
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sam_model = SAMAudio.from_pretrained(f"facebook/{model}").to(device).eval()
processor = SAMAudioProcessor.from_pretrained(f"facebook/{model}")

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 42366.71it/s]
/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Some weights of RobertaModel were not initialized from the 

In [ ]:
# Latent extraction
def get_latent(batch, prompt="A bird chirping"):
  audios = [example['path'] for example in batch["audio"]]
  descriptions = [prompt for example in batch["audio"]]

  inputs = processor(audios=audios, descriptions=descriptions).to(device) # type: ignore

  with torch.inference_mode():
    latents = sam_model.get_target_latents(inputs)

    
  embeddings_list = []
  for latent in latents:
    latent = latent[0] if latent.dim() == 3 else latent
    mean_latent = latent.mean(dim = -1)
    max_latent = latent.max(dim = -1)[0]

    # moves this back to cpu memory
    combined = torch.cat([mean_latent, max_latent], dim = -1).cpu()
    embeddings_list.append(combined)


  del inputs, latents

  return {"embedding": [e.numpy() for e in embeddings_list]}

In [9]:
# Embedding settings
split = "train"
num_clips = 100
save_dir = "/home/s.dalal.334/SAM/embeddings/"

In [10]:
clips_to_embed = data[split].select(range(num_clips))

In [ ]:
all_embeddings = tqdm(clips_to_embed.map(get_latent, batched=True, batch_size=4, load_from_cache_file=False, keep_in_memory=False, fn_kwargs={"prompt": "A bird chirping"}))
del model

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


RuntimeError: The following operation failed in the TorchScript interpreter.
Traceback of TorchScript (most recent call last):
  File "/home/s.dalal.334/SAM/.venv/lib/python3.11/site-packages/dacvae/nn/layers.py", line 22, in snake
    shape = x.shape
    x = x.reshape(shape[0], shape[1], -1)
    x = x + (alpha + 1e-9).reciprocal() * torch.sin(alpha * x).pow(2)
                                          ~~~~~~~~~ <--- HERE
    x = x.reshape(shape)
    return x
RuntimeError: CUDA out of memory. Tried to allocate 9.55 GiB. GPU 0 has a total capacity of 47.40 GiB of which 2.49 GiB is free. Including non-PyTorch memory, this process has 44.90 GiB memory in use. Of the allocated memory 43.66 GiB is allocated by PyTorch, and 957.26 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#e

In [ ]:
save_path = save_dir + "bird_latents_dataset.pt"

In [ ]:
torch.save({
  "embeddings": all_embeddings,
  "metadata": {
    "feature_order": "mean_then_max",
    "dim": 256
  }
})